# Power Demand Forecasting Pipeline
This notebook walks through the 7 steps of a Machine Learning pipeline for predicting power demand based on synthetic historical usage data and weather features.

## 1. Import the Data
Since we are generating a synthetic dataset, we'll create a DataFrame with historical power consumption, temperature, humidity, and time-based features.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Generate synthetic daily data for 2 years
np.random.seed(42)
dates = pd.date_range(start='2022-01-01', periods=730, freq='D')

# Features: Temperature (C), Humidity (%), Day of week
temperature = np.random.normal(25, 10, size=730) + np.sin(np.arange(730) * 2 * np.pi / 365) * 10
humidity = np.random.normal(60, 15, size=730)
day_of_week = dates.dayofweek
is_weekend = (day_of_week >= 5).astype(int)

# Target: Power Demand (kWh) - depends on temp, weekend, and base load
base_load = 500
power_demand = base_load + (np.abs(temperature - 22) * 15) - (is_weekend * 100) + np.random.normal(0, 30, size=730)

# Introduce some missing values to demonstrate data cleaning
missing_indices = np.random.choice(730, size=20, replace=False)
power_demand[missing_indices] = np.nan

df = pd.DataFrame({
    'Date': dates,
    'Temperature_C': temperature,
    'Humidity': humidity,
    'DayOfWeek': day_of_week,
    'IsWeekend': is_weekend,
    'Power_Demand_kWh': power_demand
})

print(f"Generated dataset with {df.shape[0]} rows.")
df.head()

## 2. Clean the Data
We need to handle the missing values in our target variable (`Power_Demand_kWh`). We'll use forward fill or interpolation.

In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())

# Interpolate missing power demand values linearly
df['Power_Demand_kWh'] = df['Power_Demand_kWh'].interpolate(method='linear')

print("\nMissing values after cleaning:")
print(df.isnull().sum())

# Set date as index
df.set_index('Date', inplace=True)

## 3. Split the Data into Training/Test Sets
We'll use an 80/20 split for our data. Because this is time-series-like data, we should ideally split chronologically, but a standard train/test split works for this regression example.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Power_Demand_kWh'])
y = df['Power_Demand_kWh']

# 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

print(f"Training Set: {X_train.shape[0]} samples")
print(f"Testing Set: {X_test.shape[0]} samples")

## 4. Create a Model
We will use a `RandomForestRegressor` from scikit-learn, which is robust and handles non-linear relationships well (like Temperature vs. Demand).

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Initialize the model
model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
print(model)

## 5. Train the Model
We fit the Random Forest model to our training data.

In [ ]:
import time

start_time = time.time()
model.fit(X_train, y_train)
end_time = time.time()

print(f"Model training completed in {end_time - start_time:.2f} seconds.")

## 6. Make Predictions
Using the trained model, we predict the power demand for our held-out test set.

In [ ]:
y_pred = model.predict(X_test)

# Display the first few predictions vs actual values
results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
results.head(10)

## 7. Evaluate and Improve
We calculate metrics like Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE) to evaluate performance, and visualize the results. To improve, one might tune hyperparameters (e.g., Grid Search) or add more features (like holidays).

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f} kWh")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} kWh")
print(f"R-squared (R2) Score: {r2:.4f}")

# Plot Actual vs Predicted
plt.figure(figsize=(14, 6))
plt.plot(results.index, results['Actual'], label='Actual Demand', alpha=0.7)
plt.plot(results.index, results['Predicted'], label='Predicted Demand', alpha=0.9)
plt.title('Power Demand: Actual vs Predicted (Test Set)')
plt.xlabel('Date')
plt.ylabel('Demand (kWh)')
plt.legend()
plt.grid(True)
plt.show()